# 20_transformation / 22__derived_metrics

Reads from the clean `financials` fact table and computes:
- **Margins** — Gross, Operating, Net (as %)
- **Free Cash Flow** — Operating CF − CapEx
- **YoY Growth** — Revenue, Net Income, Operating CF, FCF
- **Leverage** — Debt/Equity, Debt/Assets
- **Liquidity** — Current Ratio
- **Valuation** — P/E, P/S, P/FCF, EV, EV/EBITDA (requires `market_data` table)

Output table: `{catalog}.{schema}.financials_metrics`  
Long format: `ticker | company | year | metric | value`

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/00_config/01__tickers"

In [0]:
# Load ticker universe from main.config.tickers
tickers_df = spark.table(f"{CATALOG}.config.tickers")
ACTIVE_TICKERS = [row.ticker for row in tickers_df.select("ticker").collect()]
print(f"✓ Config loaded — {len(ACTIVE_TICKERS)} active tickers")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

full_table  = f"{CATALOG}.{SCHEMA}.{TABLE}"
metrics_tbl = f"{CATALOG}.{SCHEMA}.financials_metrics"
market_tbl  = f"{CATALOG}.{SCHEMA}.market_data"

## 1. Pull concepts into a wide pivot

In [0]:
needed_concepts = [
    # Income Statement
    ("Income Statement", "Revenue"),
    ("Income Statement", "Revenue (contract)"),
    ("Income Statement", "Gross Profit"),
    ("Income Statement", "Operating Income"),
    ("Income Statement", "Net Income"),
    # Balance Sheet
    ("Balance Sheet",    "Total Current Assets"),
    ("Balance Sheet",    "Total Current Liabilities"),
    ("Balance Sheet",    "Total Assets"),
    ("Balance Sheet",    "Total Liabilities"),
    ("Balance Sheet",    "Total Stockholders Equity"),
    ("Balance Sheet",    "Long-term Debt"),
    ("Balance Sheet",    "Short-term Debt"),
    ("Balance Sheet",    "Cash & Equivalents"),
    ("Balance Sheet",    "Short-term Investments"),
    # Cash Flow
    ("Cash Flow",        "Operating Cash Flow"),
    ("Cash Flow",        "CapEx"),
    ("Cash Flow",        "Depreciation & Amortization"),
]

condition = F.lit(False)
for stmt, concept in needed_concepts:
    condition = condition | (
        (F.col("stmt") == stmt) & (F.col("concept") == concept)
    )

raw = (
    spark.table(full_table)
    .filter(condition)
    .select("ticker", "company", "year", "stmt", "concept", "value")
)

wide = (
    raw
    .groupBy("ticker", "company", "year")
    .pivot("concept")
    .agg(F.first("value"))
)

# Coalesce the two Revenue columns (companies use different XBRL tags)
if "Revenue" in wide.columns and "Revenue (contract)" in wide.columns:
    wide = wide.withColumn(
        "Revenue",
        F.coalesce(F.col("Revenue"), F.col("Revenue (contract)"))
    ).drop("Revenue (contract)")

print(f"Wide table: {wide.count()} rows × {len(wide.columns)} columns")

## 2. Compute margins, FCF, leverage, liquidity

In [0]:
def safe_div(num, den):
    """Null-safe division."""
    return F.when(F.col(den).isNotNull() & (F.col(den) != 0),
                  F.col(num) / F.col(den)).otherwise(F.lit(None))

metrics_wide = (
    wide
    # ── Margins (%) ───────────────────────────────────────────────────────────
    .withColumn("Gross Margin %",     safe_div("Gross Profit",    "Revenue") * 100)
    .withColumn("Operating Margin %", safe_div("Operating Income","Revenue") * 100)
    .withColumn("Net Margin %",       safe_div("Net Income",      "Revenue") * 100)

    # ── Free Cash Flow ────────────────────────────────────────────────────────
    .withColumn("Free Cash Flow",
        F.when(F.col("Operating Cash Flow").isNotNull() & F.col("CapEx").isNotNull(),
               F.col("Operating Cash Flow") - F.col("CapEx"))
    )

    # ── Leverage ──────────────────────────────────────────────────────────────
    .withColumn("Total Debt",
        F.coalesce(F.col("Long-term Debt"), F.lit(0)) +
        F.coalesce(F.col("Short-term Debt"), F.lit(0))
    )
    .withColumn("Debt / Equity",  safe_div("Total Debt", "Total Stockholders Equity"))
    .withColumn("Debt / Assets",  safe_div("Total Debt", "Total Assets"))

    # ── Liquidity ─────────────────────────────────────────────────────────────
    .withColumn("Current Ratio",  safe_div("Total Current Assets", "Total Current Liabilities"))

    # ── FCF Margin ────────────────────────────────────────────────────────────
    .withColumn("FCF Margin %",   safe_div("Free Cash Flow", "Revenue") * 100)
)

## 3. YoY growth (window function)

In [0]:
grow_cols = ["Revenue", "Net Income", "Operating Cash Flow", "Free Cash Flow"]
w = Window.partitionBy("ticker").orderBy("year")

for col in grow_cols:
    if col in metrics_wide.columns:
        lag_col = f"_lag_{col}"
        metrics_wide = (
            metrics_wide
            .withColumn(lag_col, F.lag(F.col(col)).over(w))
            .withColumn(
                f"{col} YoY %",
                F.when(
                    F.col(lag_col).isNotNull() & (F.col(lag_col) != 0),
                    ((F.col(col) - F.col(lag_col)) / F.abs(F.col(lag_col))) * 100
                )
            )
            .drop(lag_col)
        )

## 4. Melt to long format

In [0]:
raw_concept_cols = [
    "Revenue", "Gross Profit", "Operating Income", "Net Income",
    "Total Current Assets", "Total Current Liabilities", "Total Assets",
    "Total Liabilities", "Total Stockholders Equity", "Long-term Debt",
    "Short-term Debt", "Operating Cash Flow", "CapEx", "Total Debt",
    "Cash & Equivalents", "Short-term Investments", "Depreciation & Amortization",
]

id_cols     = ["ticker", "company", "year"]
metric_cols = [c for c in metrics_wide.columns
               if c not in id_cols and c not in raw_concept_cols]

print(f"Computed metrics: {metric_cols}")

stack_expr = ", ".join([f"'{c}', `{c}`" for c in metric_cols])
n = len(metric_cols)

metrics_long = metrics_wide.selectExpr(
    "ticker", "company", "year",
    f"stack({n}, {stack_expr}) as (metric, value)"
).filter(F.col("value").isNotNull())

print(f"Long metrics table: {metrics_long.count()} rows")

## 5. Write to Delta (MERGE — idempotent)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {metrics_tbl} (
        ticker  STRING  NOT NULL,
        company STRING,
        year    INT     NOT NULL,
        metric  STRING  NOT NULL,
        value   DOUBLE
    )
    USING DELTA
    PARTITIONED BY (ticker)
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")

metrics_long.createOrReplaceTempView("incoming_metrics")

spark.sql(f"""
    MERGE INTO {metrics_tbl} AS target
    USING incoming_metrics AS source
    ON  target.ticker = source.ticker
    AND target.year   = source.year
    AND target.metric = source.metric

    WHEN MATCHED AND target.value != source.value THEN
        UPDATE SET target.value = source.value, target.company = source.company

    WHEN NOT MATCHED THEN
        INSERT (ticker, company, year, metric, value)
        VALUES (source.ticker, source.company, source.year, source.metric, source.value)
""")

print(f"✓ Merged into {metrics_tbl}")

## 6. Valuation metrics
Joins with `market_data` to compute P/E, P/S, P/FCF, EV, EV/EBITDA.  
Requires `12__fetch_market_data` to have run first.

In [0]:
try:
    mkt = (
        spark.table(market_tbl)
        .select("ticker", "year", "market_cap")
        .filter(F.col("market_cap").isNotNull())
    )
    print(f"✓ market_data loaded — {mkt.count():,} rows")
except Exception:
    print("⚠ market_data not found — skipping valuation metrics.")
    print("  Run 12__fetch_market_data first, then re-run this notebook.")
    mkt = None

In [0]:
if mkt is not None:

    # Pull raw concepts needed for valuation
    val_concepts = (
        spark.table(full_table)
        .filter(
            (
                (F.col("stmt") == "Income Statement") &
                (F.col("concept").isin("Net Income", "Revenue", "Operating Income"))
            ) | (
                (F.col("stmt") == "Cash Flow") &
                (F.col("concept").isin(
                    "Operating Cash Flow", "CapEx", "Depreciation & Amortization"
                ))
            ) | (
                (F.col("stmt") == "Balance Sheet") &
                (F.col("concept").isin(
                    "Cash & Equivalents", "Short-term Investments",
                    "Long-term Debt", "Short-term Debt"
                ))
            )
        )
        .groupBy("ticker", "company", "year")
        .pivot("concept")
        .agg(F.first("value"))
    )

    def safe_div_col(num_col, den_col):
        return F.when(
            den_col.isNotNull() & (den_col != 0),
            num_col / den_col
        ).otherwise(F.lit(None))

    val_wide = (
        mkt.join(val_concepts, on=["ticker", "year"], how="left")
        .withColumn("_fcf",
            F.when(
                F.col("Operating Cash Flow").isNotNull() & F.col("CapEx").isNotNull(),
                F.col("Operating Cash Flow") - F.col("CapEx")
            )
        )
        .withColumn("_total_debt",
            F.coalesce(F.col("Long-term Debt"),  F.lit(0)) +
            F.coalesce(F.col("Short-term Debt"), F.lit(0))
        )
        .withColumn("_cash",
            F.coalesce(F.col("Cash & Equivalents"),     F.lit(0)) +
            F.coalesce(F.col("Short-term Investments"),  F.lit(0))
        )
        .withColumn("_ev",
            F.col("market_cap") + F.col("_total_debt") - F.col("_cash")
        )
        .withColumn("_ebitda",
            F.when(
                F.col("Operating Income").isNotNull(),
                F.col("Operating Income") +
                F.coalesce(F.col("Depreciation & Amortization"), F.lit(0))
            )
        )
    )

    val_metric_cols = ["P/E", "P/S", "P/FCF", "EV", "EV/EBITDA"]

    val_metrics = (
        val_wide
        .withColumn("P/E",       safe_div_col(F.col("market_cap"), F.col("Net Income")))
        .withColumn("P/S",       safe_div_col(F.col("market_cap"), F.col("Revenue")))
        .withColumn("P/FCF",     safe_div_col(F.col("market_cap"), F.col("_fcf")))
        .withColumn("EV",        F.col("_ev"))
        .withColumn("EV/EBITDA", safe_div_col(F.col("_ev"),        F.col("_ebitda")))
    )

    val_long = (
        val_metrics
        .selectExpr(
            "ticker", "company", "year",
            f"stack({len(val_metric_cols)}, "
            + ", ".join([f"'{c}', `{c}`" for c in val_metric_cols])
            + ") as (metric, value)"
        )
        .filter(F.col("value").isNotNull())
        .filter(
            (F.col("metric") == "EV") |
            F.col("value").between(-500, 500)
        )
    )

    print(f"Valuation metrics rows: {val_long.count():,}")

    val_long.createOrReplaceTempView("incoming_valuation")

    spark.sql(f"""
        MERGE INTO {metrics_tbl} AS target
        USING incoming_valuation AS source
        ON  target.ticker = source.ticker
        AND target.year   = source.year
        AND target.metric = source.metric

        WHEN MATCHED AND target.value != source.value THEN
            UPDATE SET target.value = source.value, target.company = source.company

        WHEN NOT MATCHED THEN
            INSERT (ticker, company, year, metric, value)
            VALUES (source.ticker, source.company, source.year, source.metric, source.value)
    """)

    print(f"✓ Valuation metrics merged into {metrics_tbl}")

## 7. Preview

In [0]:
spark.sql(f"""
    SELECT ticker, year, metric, ROUND(value, 2) AS value
    FROM {metrics_tbl}
    WHERE ticker = 'AAPL'
      AND metric IN ('Gross Margin %', 'Net Margin %', 'Free Cash Flow',
                     'Current Ratio', 'Revenue YoY %',
                     'P/E', 'P/S', 'P/FCF', 'EV/EBITDA')
    ORDER BY metric, year
""").display()